<div style="background: linear-gradient(135deg, #7B4F00 0%, #f5a623 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #fff3d6; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #fff3d6; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: white; font-size: 1.05em; font-weight: 600;">&#9733; Bonus Chapter &nbsp;&middot;&nbsp; From Notebook to Production</span>
  </div>
</div>
<div style="background: #fff8ec; border-left: 5px solid #f5a623; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>Once a model is trained and saved, how do you make it available to other systems, teams, and users?
  This chapter takes the ModelPipeline .pth file from Chapter 3 and walks through the full journey
  from notebook to a live prediction endpoint.</em>
</div>

## What This Chapter Covers

| Section | Topics | Exercise |
|---------|--------|----------|
| B.1 Why Notebooks Are Not Enough | The gap between a notebook and a deployed model · What production means · Who consumes model output | Map a model to a realistic deployment scenario |
| B.2 Serving Predictions with FastAPI | What an API is · Loading ModelPipeline in FastAPI · Writing a /predict endpoint · Testing locally | Build a /predict endpoint that accepts JSON and returns a prediction |
| B.3 Containerising with Docker | Why containers solve the "works on my machine" problem · Writing a Dockerfile · Building and running locally | Containerise the app; send a request to the running container |
| B.4 Deploying to the Cloud | Free deployment options · Pushing to Render · Testing the live endpoint from anywhere | Deploy the container; call the live endpoint from a notebook |
| B.5 Updating a Deployed Model | Retraining · Saving v2 · Redeploying · Version naming | Retrain, save as v2, redeploy, confirm predictions changed |

> **Prerequisites:** Chapters 1–3 must be completed.
> The `california_pipeline_v1.pth` file from Chapter 3 is the model we deploy throughout.
>
> **Tools needed (all free):**
> - Docker Desktop — download at [docker.com/products/docker-desktop](https://www.docker.com/products/docker-desktop)
> - A free Render account — [render.com](https://render.com)
> - A free GitHub account — [github.com](https://github.com)
>
> **Windows users:** see the Windows setup note in Section B.3 before installing Docker.

---
## Setup

Run this cell first. It installs FastAPI and the tools needed for local testing.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bonus Chapter — Setup
# These packages are used for local API testing inside the notebook.
# FastAPI and uvicorn are also what runs inside the Docker container.
# ─────────────────────────────────────────────────────────────────────────────

!pip install --quiet fastapi uvicorn[standard] pydantic httpx torch scikit-learn dill numpy

import torch, numpy as np, dill, json, textwrap
print("Setup complete.")

---
## Where We Are Coming From

In Chapter 3 you trained a regression network on the California Housing dataset
and saved everything into a single file using `ModelPipeline`:

```
california_pipeline_v1.pth
```

That file contains the model weights, the fitted scaler, the feature names,
and the architecture config — everything needed to reproduce a prediction
on any machine that has PyTorch installed.

**The problem:** right now, only you can use this model. To get a prediction,
someone has to open your notebook, find the right cell, and run it. That is not
practical for a colleague, a web application, or an automated system.

**This chapter answers the question every chapter has been building toward:**

> *Once a model is trained and saved, how do you make it available
> to other systems, teams, and users?*

By the end of this chapter, your model will be reachable at a URL like:

```
POST https://house-predictor.onrender.com/predict
Body: {"MedInc": 5.2, "HouseAge": 25, "AveRooms": 6.1, ...}

Response: {"predicted_house_value": "$248,000"}
```

Any application — a web app, a dashboard, a colleague's notebook, a mobile app —
can call that URL and get a prediction back, without any Python knowledge.

---
# B.1 Why Notebooks Are Not Enough

## What a notebook cannot do

A Jupyter notebook is a document. It requires a human to open it, run it,
and read it. No other program can send data to a notebook and receive a result.

A **deployed model** runs as a service — waiting for requests, processing them
instantly, and returning results — without any human involvement.

| | Notebook | Deployed model |
|-|----------|----------------|
| Who can use it | Only the person who has it open | Any application or user with the URL |
| How to get a prediction | Open the notebook, run cells manually | Send a web request, receive JSON back |
| Works while you are asleep | No | Yes |
| Can a web app or dashboard call it | No | Yes |
| Requires Python knowledge to use | Yes | No |

## Who consumes the model output

Before deploying a model, it is worth asking: who will actually call this endpoint?
The answer shapes how you design the API.

| Consumer | What they send | What they expect back |
|----------|---------------|----------------------|
| A web application | Form data in JSON | A prediction + confidence |
| A scheduled batch job | A file of 10,000 rows | A file of predictions |
| A colleague's notebook | A `requests.post()` call | A JSON response |
| A mobile app | Sensor readings | A classification label |
| A dashboard | The latest data point | A forecast value |

## The production stack

Getting from a `.pth` file to a live endpoint uses three tools:

```
.pth file        FastAPI           Docker            Cloud
(your model) --> (web server) --> (container) -->  (public URL)

Holds the        Wraps the         Packages          Runs your
trained          model in          everything        container on
pipeline         a web API         together          the internet
```

| Tool | What it does | Analogy |
|------|-------------|---------|
| **FastAPI** | Turns a Python function into a web endpoint | A receptionist who takes incoming requests and routes them to the model |
| **Docker** | Packages your code and all its dependencies into one portable box | A sealed shipping container — identical contents, runs anywhere |
| **Render** | Runs your Docker container on a server in the cloud | A warehouse that operates your shipping containers for you |

### 📝 Exercise B.1 — Map a Model to a Deployment Scenario

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise B.1
#
# For each model below, identify a realistic deployment scenario.
# Write your answers as comments.
#
# For each model, answer:
#   Consumer  : who or what will call this endpoint?
#   Input     : what data will they send?
#   Output    : what should the endpoint return?
#   Frequency : how often will it be called? (real-time / hourly / nightly)
#
# Example (Chapter 3 — House Price Regression):
#   Consumer  : a property listing website
#   Input     : house features (rooms, location, age, income of area)
#   Output    : predicted median price + estimated range
#   Frequency : real-time (every time a new listing is entered)
# ─────────────────────────────────────────────────────────────────────────────

scenarios = {
    "Chapter 4 -- Customer Churn Classifier": {
        "Consumer":  "?",
        "Input":     "?",
        "Output":    "?",
        "Frequency": "?",
    },
    "Chapter 5 -- Product Defect CNN": {
        "Consumer":  "?",
        "Input":     "?",
        "Output":    "?",
        "Frequency": "?",
    },
    "Chapter 6 -- CO2 Demand Forecaster": {
        "Consumer":  "?",
        "Input":     "?",
        "Output":    "?",
        "Frequency": "?",
    },
}

for model, answers in scenarios.items():
    print(f"\n{model}")
    for k, v in answers.items():
        print(f"  {k:<12}: {v}")

---
# B.2 Serving Predictions with FastAPI

## What an API is — in plain language

An **API** (Application Programming Interface) is a way for two programs to talk
to each other over the internet. You already used one in Chapter 7 when you called
`client.chat.completions.create(...)` — that was an API call to an LLM provider.

A **REST API** uses standard web requests — the same kind your browser makes
when you load a webpage. Every interaction has two parts:

```
REQUEST (caller sends):
  Method : POST   <-- "I am sending you data"
  URL    : /predict
  Body   : {"MedInc": 5.2, "HouseAge": 25, "AveRooms": 6.1, ...}

RESPONSE (server sends back):
  Status : 200 OK
  Body   : {"predicted_house_value": "$248,000"}
```

**FastAPI** is a Python library that lets you write an API endpoint as a regular
Python function. You write the logic; FastAPI handles everything else — receiving
the request, validating the input, formatting the response, and serving it over HTTP.

---

## The project structure

Before writing any code, let us agree on the file structure.
Everything for deployment lives in one folder:

```
house-predictor/
    app.py                       <- the FastAPI application  (we write this now)
    california_pipeline_v1.pth   <- the trained model from Chapter 3
    requirements.txt             <- Python dependencies      (written in B.3)
    Dockerfile                   <- container instructions   (written in B.3)
```

All four files will exist by the end of Section B.3.

## Writing `app.py`

The cell below writes `app.py` to disk. Read through every section —
each numbered comment corresponds to one responsibility of the application.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write app.py -- the complete FastAPI application
#
# We write the file line-by-line to avoid any quoting issues in the notebook.
# In practice you would write this directly in a text editor (VS Code, Notepad).
# ─────────────────────────────────────────────────────────────────────────────

app_lines = [
    "import torch",
    "import numpy as np",
    "import dill",
    "from fastapi import FastAPI, HTTPException",
    "from pydantic import BaseModel",
    "",
    "# ── 1. ModelPipeline (load-only) ─────────────────────────────────────────",
    "# We need the ModelPipeline class in scope to call .load().",
    "# This is the same class from Chapter 3 -- pasted here so app.py is",
    "# self-contained and does not depend on any external training code.",
    "",
    "class ModelPipeline:",
    "    # Minimal load-only version of the Chapter 3 ModelPipeline.",
    "    def __init__(self):",
    "        self.model = None",
    "        self.preprocessor = None",
    "        self.feature_names = []",
    "        self.model_config = {}",
    "        self.training_history = {}",
    "",
    "    @classmethod",
    "    def load(cls, path, device=None):",
    '        device = device or torch.device("cpu")',
    "        checkpoint = torch.load(path, map_location=device, weights_only=False)",
    "        obj = cls()",
    '        ModelClass = dill.loads(checkpoint["model_class_bytes"])',
    '        obj.model = ModelClass(**checkpoint["model_config"])',
    '        obj.model.load_state_dict(checkpoint["state_dict"])',
    "        obj.model.eval()",
    '        obj.preprocessor     = checkpoint.get("preprocessor")',
    '        obj.feature_names    = checkpoint.get("feature_names", [])',
    '        obj.model_config     = checkpoint.get("model_config", {})',
    '        obj.training_history = checkpoint.get("training_history", {})',
    "        return obj",
    "",
    "    def predict(self, X_raw, device=None):",
    '        device = device or torch.device("cpu")',
    "        X = self.preprocessor.transform(X_raw) if self.preprocessor else X_raw",
    "        t = torch.tensor(X, dtype=torch.float32).to(device)",
    "        with torch.no_grad():",
    "            return self.model(t).cpu().numpy()",
    "",
    "# ── 2. Load the model once at startup ────────────────────────────────────",
    "# This runs ONCE when the server starts, not on every request.",
    "# Loading once and keeping the pipeline in memory is far more efficient.",
    "",
    'pipeline = ModelPipeline.load("california_pipeline_v1.pth")',
    'print(f"Model loaded. Features: {pipeline.feature_names}")',
    "",
    "# ── 3. Input schema ───────────────────────────────────────────────────────",
    "# BaseModel defines exactly what JSON the /predict endpoint accepts.",
    "# FastAPI uses this to validate every incoming request automatically.",
    "# If a required field is missing, FastAPI returns a clear 422 error",
    "# before your code even runs -- no manual validation needed.",
    "",
    "class HouseFeatures(BaseModel):",
    "    MedInc:     float   # Median income in block group (x $10,000)",
    "    HouseAge:   float   # Median house age (years)",
    "    AveRooms:   float   # Average rooms per household",
    "    AveBedrms:  float   # Average bedrooms per household",
    "    Population: float   # Block group population",
    "    AveOccup:   float   # Average household members",
    "    Latitude:   float   # Block group latitude",
    "    Longitude:  float   # Block group longitude",
    "",
    "# ── 4. Create the FastAPI app ─────────────────────────────────────────────",
    "app = FastAPI(",
    '    title       = "House Price Predictor",',
    '    description = "Predicts California median house prices. Model from Chapter 3.",',
    '    version     = "1.0.0",',
    ")",
    "",
    "# ── 5. Health-check endpoint ──────────────────────────────────────────────",
    "# GET /  returns {status: ok} so cloud platforms can confirm the server is alive.",
    "# Always useful for quickly checking that a deployment succeeded.",
    "",
    '@app.get("/")',
    "def health_check():",
    '    return {"status": "ok", "model": "california_pipeline_v1"}',
    "",
    "# ── 6. Prediction endpoint ────────────────────────────────────────────────",
    "# POST /predict  accepts JSON matching HouseFeatures, runs the pipeline,",
    "# and returns the predicted house value.",
    "",
    '@app.post("/predict")',
    "def predict(features: HouseFeatures):",
    "    try:",
    "        X = np.array([[",
    "            features.MedInc,   features.HouseAge, features.AveRooms,",
    "            features.AveBedrms,features.Population,features.AveOccup,",
    "            features.Latitude, features.Longitude,",
    "        ]])",
    "        raw = pipeline.predict(X)",
    "        # Model trained on log-transformed targets (Chapter 3) -- convert back",
    "        usd = float(np.expm1(raw[0][0]) * 100_000)",
    "        return {",
    '            "predicted_house_value_usd":     round(usd, 2),',
    '            "predicted_house_value_display": f"${usd:,.0f}",',
    "        }",
    "    except Exception as e:",
    "        raise HTTPException(status_code=400, detail=str(e))",
]

with open("app.py", "w") as f:
    f.write("\n".join(app_lines))

print("app.py written successfully.")
print()
print("File structure so far:")
print("  app.py                       <- FastAPI application")
print("  california_pipeline_v1.pth   <- trained model from Chapter 3")
print()
print("Still needed:")
print("  requirements.txt             <- written in Section B.3")
print("  Dockerfile                   <- written in Section B.3")

## Testing the API locally — inside this notebook

Before packaging anything, we test the API right here. The cell below:
1. Starts the FastAPI server as a background process
2. Sends it three test requests using the `httpx` library
3. Stops the server

This confirms the API logic is correct before we touch Docker.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Start the server, run tests, stop the server
#
# uvicorn is the web server that runs FastAPI applications.
# We launch it as a background subprocess so this cell can continue running.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, time, httpx, json

# Start server in background
server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)   # wait for startup

print("=" * 55)
print("Local API Tests")
print("=" * 55)

# ── Test 1: health check ──────────────────────────────────────────────────────
r = httpx.get("http://127.0.0.1:8000/")
print(f"\nTest 1 -- Health check")
print(f"  Status : {r.status_code}")
print(f"  Body   : {r.json()}")

# ── Test 2: valid prediction ──────────────────────────────────────────────────
sample = {
    "MedInc": 5.2, "HouseAge": 25.0, "AveRooms": 6.1, "AveBedrms": 1.1,
    "Population": 1200.0, "AveOccup": 3.1, "Latitude": 34.2, "Longitude": -118.4,
}
r2 = httpx.post("http://127.0.0.1:8000/predict", json=sample)
print(f"\nTest 2 -- Valid prediction")
print(f"  Input  : MedInc={sample['MedInc']}, HouseAge={sample['HouseAge']}, ...")
print(f"  Status : {r2.status_code}")
print(f"  Output : {r2.json()}")

# ── Test 3: missing field (should return 422) ─────────────────────────────────
r3 = httpx.post("http://127.0.0.1:8000/predict", json={"MedInc": 5.2})
print(f"\nTest 3 -- Missing fields (expect 422 Validation Error)")
print(f"  Status : {r3.status_code}  <- 422 means FastAPI caught the bad input")

# ── Test 4: batch of predictions ──────────────────────────────────────────────
batch = [
    {"MedInc": 8.5, "HouseAge": 10, "AveRooms": 7.2, "AveBedrms": 1.0,
     "Population": 900,  "AveOccup": 2.8, "Latitude": 37.8, "Longitude": -122.4},
    {"MedInc": 2.1, "HouseAge": 40, "AveRooms": 4.1, "AveBedrms": 1.3,
     "Population": 2100, "AveOccup": 3.8, "Latitude": 34.0, "Longitude": -118.2},
    {"MedInc": 5.2, "HouseAge": 25, "AveRooms": 6.1, "AveBedrms": 1.1,
     "Population": 1200, "AveOccup": 3.1, "Latitude": 34.2, "Longitude": -118.4},
]
print(f"\nTest 4 -- Batch of 3 predictions")
print(f"  {'MedInc':>8}  {'HouseAge':>9}  {'Prediction':>18}")
print("  " + "-" * 40)
for inp in batch:
    r = httpx.post("http://127.0.0.1:8000/predict", json=inp)
    pred = r.json().get("predicted_house_value_display", "error")
    print(f"  {inp['MedInc']:>8.1f}  {inp['HouseAge']:>9.0f}  {pred:>18}")

# Stop the server
server.terminate()
print(f"\nAll tests passed. Server stopped.")

> **What each status code means:**
>
> | Status | Meaning |
> |--------|---------|
> | `200 OK` | Request succeeded, prediction returned |
> | `422 Unprocessable Entity` | Input validation failed — FastAPI rejected a bad request automatically |
> | `400 Bad Request` | Prediction logic raised an error (e.g. wrong data types) |
>
> A `422` on Test 3 is the *correct* behaviour — it means FastAPI's automatic
> validation is working and bad data never reaches your model.

### 📝 Exercise B.2 — Add a second endpoint

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise B.2 -- Add a /predict/batch endpoint
#
# The current /predict endpoint handles one house at a time.
# Add a second endpoint /predict/batch that:
#   - Accepts a JSON list of HouseFeatures objects
#   - Returns a list of predictions, one per input
#
# Skeleton:
#
#   class BatchRequest(BaseModel):
#       houses: list[HouseFeatures]
#
#   @app.post("/predict/batch")
#   def predict_batch(request: BatchRequest):
#       results = []
#       for house in request.houses:
#           # call the same predict() logic
#           results.append(...)
#       return {"predictions": results}
#
# Steps:
#   1. Add the BatchRequest class and predict_batch() function to app.py
#   2. Restart the server (re-run the start cell above)
#   3. Send a batch request using httpx and verify all predictions return
#
# Reflection:
#   - Why is a batch endpoint more efficient than calling /predict 100 times?
#   - What would you change in the response format to make it easier for a
#     downstream application to match predictions back to their input rows?
# ─────────────────────────────────────────────────────────────────────────────

print("Exercise B.2 -- add /predict/batch to app.py and test it.")

---
# B.3 Containerising with Docker

## The "works on my machine" problem

The API works perfectly on your machine. Now imagine sending your code to a
colleague or deploying it to a cloud server. They will need:

- The same Python version
- The same library versions (torch, fastapi, dill, ...)
- The same file paths
- The same operating system behaviour

Getting all of this to match across machines is notoriously painful.
This is the **"works on my machine" problem** — and it is why Docker exists.

---

## What a container is

A **container** is a sealed, self-contained package that includes:

```
Your container
+-----------------------------------------+
|  Your code       app.py                 |
|  Your model      california_pipeline_v1.pth |
|  Python 3.10     (exact version)        |
|  All libraries   (exact versions)       |
|  Run command     uvicorn app:app ...    |
+-----------------------------------------+
```

You build the container once on your machine. That exact container — with
everything inside it — runs identically on any cloud server in the world.

```
WITHOUT Docker:
  Your machine   --> Works perfectly
  Cloud server   --> Python version mismatch, missing library, crash

WITH Docker:
  Your machine   --> Build container once
  Cloud server   --> Run the same container --> identical behaviour
```

---

## Windows setup note

> **If you are on Windows**, Docker Desktop requires WSL2 (Windows Subsystem for Linux).
>
> **Steps (one-time, takes about 10 minutes):**
>
> 1. Open PowerShell as Administrator and run:
>    ```
>    wsl --install
>    ```
>    This installs WSL2 automatically. Restart your computer when prompted.
>
> 2. Download and install Docker Desktop from [docker.com/products/docker-desktop](https://www.docker.com/products/docker-desktop)
>    During installation, make sure "Use WSL 2 based engine" is checked (it usually is by default).
>
> 3. Start Docker Desktop. Wait for the whale icon in the system tray to stop animating
>    — that means Docker is ready.
>
> 4. Open **PowerShell** (not Command Prompt) and run:
>    ```
>    docker --version
>    ```
>    You should see something like `Docker version 24.0.5`. If you do, Docker is working.
>
> **All Docker commands in this chapter work in PowerShell on Windows.**
> You do not need Linux or a Mac.

---

## The four files

To containerise the FastAPI app you need exactly four files in one folder:

```
house-predictor/
    app.py                       <- written in Section B.2
    california_pipeline_v1.pth   <- trained model from Chapter 3
    requirements.txt             <- Python libraries to install  (next cell)
    Dockerfile                   <- instructions to build the container  (cell after)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write requirements.txt
#
# Lists every Python package the container needs.
# Pinning exact versions ensures the container builds identically every time.
# ─────────────────────────────────────────────────────────────────────────────

req_lines = [
    "fastapi==0.111.0",
    "uvicorn[standard]==0.29.0",
    "pydantic==2.7.0",
    "torch==2.2.0",
    "numpy==1.26.4",
    "scikit-learn==1.4.2",
    "dill==0.3.8",
]

with open("requirements.txt", "w") as f:
    f.write("\n".join(req_lines) + "\n")

print("requirements.txt written:")
for line in req_lines:
    print(f"  {line}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Write Dockerfile
#
# A Dockerfile is a recipe. Docker reads it top to bottom and builds
# the container layer by layer. Each instruction is one step.
# ─────────────────────────────────────────────────────────────────────────────

dockerfile_lines = [
    "# Start from a minimal Python 3.10 image.",
    "# 'slim' means no unnecessary extras -- keeps the container small.",
    "FROM python:3.10-slim",
    "",
    "# Set the working directory inside the container.",
    "# All subsequent paths are relative to /app.",
    "WORKDIR /app",
    "",
    "# Copy requirements.txt first.",
    "# Docker caches this layer -- if requirements haven't changed,",
    "# it skips reinstalling on the next build. Saves several minutes.",
    "COPY requirements.txt .",
    "",
    "# Install all Python dependencies.",
    "RUN pip install --no-cache-dir -r requirements.txt",
    "",
    "# Copy the application file and the model file.",
    "COPY app.py .",
    "COPY california_pipeline_v1.pth .",
    "",
    "# Tell Docker which port the app listens on.",
    "# This is documentation only -- it does not open the port by itself.",
    "EXPOSE 8000",
    "",
    "# The command that runs when the container starts.",
    "# --host 0.0.0.0 means 'accept connections from outside the container'.",
    'CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]',
]

with open("Dockerfile", "w") as f:
    f.write("\n".join(dockerfile_lines) + "\n")

print("Dockerfile written.")
print()
print("All four files are ready:")
print("  app.py")
print("  california_pipeline_v1.pth")
print("  requirements.txt")
print("  Dockerfile")

## Building and running the container

These commands run in a **terminal** (PowerShell on Windows, Terminal on Mac/Linux),
not inside the notebook. Open a terminal, navigate to the folder containing your
four files, and run them in order.

### Step 1 — Build the container image

```powershell
docker build -t house-predictor .
```

What this does:
- Reads the `Dockerfile`
- Downloads `python:3.10-slim` (first time only — ~150 MB)
- Installs all packages from `requirements.txt` (~5 minutes on first run)
- Copies your files in
- Tags the result as `house-predictor`

You will see output like:
```
Step 1/8 : FROM python:3.10-slim
Step 2/8 : WORKDIR /app
...
Successfully tagged house-predictor:latest
```

### Step 2 — Run the container

```powershell
docker run -p 8000:8000 house-predictor
```

What `-p 8000:8000` means: connect port 8000 on *your machine* to port 8000 *inside the container*.
Without this, the container runs in isolation and nothing can reach it.

You will see:
```
Model loaded. Features: ['MedInc', 'HouseAge', ...]
INFO:     Uvicorn running on http://0.0.0.0:8000
```

### Step 3 — Test it

Open a **second** terminal window (leave the first one running the container) and run:

**Windows PowerShell:**
```powershell
Invoke-WebRequest -Uri http://localhost:8000/ -Method GET | Select-Object -ExpandProperty Content
```

**Mac/Linux:**
```bash
curl http://localhost:8000/
```

You should get back:
```json
{"status":"ok","model":"california_pipeline_v1"}
```

For a prediction:

**Windows PowerShell:**
```powershell
$body = '{"MedInc":5.2,"HouseAge":25,"AveRooms":6.1,"AveBedrms":1.1,"Population":1200,"AveOccup":3.1,"Latitude":34.2,"Longitude":-118.4}'
Invoke-WebRequest -Uri http://localhost:8000/predict -Method POST -ContentType "application/json" -Body $body | Select-Object -ExpandProperty Content
```

**Mac/Linux:**
```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"MedInc":5.2,"HouseAge":25,"AveRooms":6.1,"AveBedrms":1.1,"Population":1200,"AveOccup":3.1,"Latitude":34.2,"Longitude":-118.4}'
```

Expected response:
```json
{"predicted_house_value_usd": 248750.0, "predicted_house_value_display": "$248,750"}
```

### Inspecting logs and stopping the container

To see what the container is doing, the terminal running `docker run` shows every
incoming request:
```
INFO:     127.0.0.1:54321 - "POST /predict HTTP/1.1" 200 OK
```

To stop the container: press `Ctrl+C` in the terminal where it is running.

> **If the container runs correctly, you have solved the "works on my machine" problem.**
> This exact container will produce identical results on any cloud server.

### 📝 Exercise B.3 — Containerise a second model

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise B.3 -- Containerise the Chapter 4 churn model
#
# Repeat the process from B.2 and B.3 for the Chapter 4 churn classifier.
# The churn model has different input features and returns a probability,
# not a dollar value.
#
# Tasks:
#   1. Create a new folder called churn-predictor/
#
#   2. Write a new app.py for the churn model:
#      - Load churn_model_v1.pth
#      - Define a ChurnFeatures BaseModel with the correct fields
#        (tenure, monthly_charges, total_charges, etc.)
#      - Write a /predict endpoint that returns:
#        {"churn_probability": 0.73, "risk": "High"}
#
#   3. Copy requirements.txt and Dockerfile into churn-predictor/
#      (only change: COPY churn_model_v1.pth in the Dockerfile)
#
#   4. Build and run:
#      docker build -t churn-predictor .
#      docker run -p 8001:8000 churn-predictor
#      (port 8001 on your machine so it doesn't clash with house-predictor)
#
#   5. Test from a Python cell:
#      import httpx
#      r = httpx.post("http://localhost:8001/predict", json={...})
#      print(r.json())
# ─────────────────────────────────────────────────────────────────────────────

print("Exercise B.3 -- containerise the churn model.")
print("Follow the steps in the comment above.")

---
# B.4 Deploying to the Cloud

## What deploying means

Right now your container runs on your laptop. Deploying means moving it to a
server on the internet so it is reachable from anywhere — without your laptop
needing to be on.

---

## Free options — no credit card required

| Platform | Free tier | How you deploy |
|----------|-----------|----------------|
| **Render** | 750 hrs/month, sleeps after 15 min idle | Connect GitHub repo → auto-deploys on every push |
| **Railway** | $5 free credit/month | Similar to Render, slightly more flexible |
| **Hugging Face Spaces** | Free CPU tier | Good for models, has a built-in UI option |

We will use **Render** — it is the simplest for a first deployment and the free
tier is generous enough for learning and demonstration purposes.

---

## Deploying to Render — step by step

No notebook cells needed for this section.
Follow these steps in your browser and terminal.

---

### Step 1 — Push your four files to GitHub

If you do not have a GitHub account, create a free one at [github.com](https://github.com).

Create a new repository called `house-predictor` (public or private, both work).

In your terminal, inside the `house-predictor/` folder:

```powershell
git init
git add app.py requirements.txt Dockerfile california_pipeline_v1.pth
git commit -m "Initial deployment"
git branch -M main
git remote add origin https://github.com/YOUR-USERNAME/house-predictor.git
git push -u origin main
```

> **Windows tip:** If `git` is not installed, download Git for Windows from
> [git-scm.com](https://git-scm.com). During install, accept all defaults.
> Then open a new PowerShell window and try again.

---

### Step 2 — Create a Render Web Service

1. Go to [render.com](https://render.com) and sign up (free, no credit card).
2. Click **New +** → **Web Service**.
3. Connect your GitHub account when prompted.
4. Select the `house-predictor` repository.

---

### Step 3 — Configure the service

| Setting | Value |
|---------|-------|
| Name | house-predictor |
| Environment | **Docker** |
| Region | Closest to you |
| Instance Type | **Free** |
| Port | 8000 |

Leave all other settings at their defaults. Click **Create Web Service**.

Render will:
1. Pull your code from GitHub
2. Build the Docker container (takes 3–8 minutes on first build)
3. Start the container
4. Give you a live URL

---

### Step 4 — Get your live URL

When the deployment is complete, Render shows a green "Live" indicator and a URL:

```
https://house-predictor-xxxx.onrender.com
```

Open that URL in your browser. You should see:
```json
{"status":"ok","model":"california_pipeline_v1"}
```

**Your model is now live on the internet.**

---

## Calling your live endpoint from a notebook

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Call your deployed endpoint from this notebook
#
# Replace RENDER_URL with the actual URL Render gave you.
# ─────────────────────────────────────────────────────────────────────────────

import httpx

RENDER_URL = "https://house-predictor-xxxx.onrender.com"   # <- replace this

# ── Health check ──────────────────────────────────────────────────────────────
r = httpx.get(f"{RENDER_URL}/")
print(f"Health check  : {r.status_code}  {r.json()}")

# ── Single prediction ─────────────────────────────────────────────────────────
sample = {
    "MedInc": 5.2, "HouseAge": 25.0, "AveRooms": 6.1, "AveBedrms": 1.1,
    "Population": 1200.0, "AveOccup": 3.1, "Latitude": 34.2, "Longitude": -118.4,
}
r2 = httpx.post(f"{RENDER_URL}/predict", json=sample)
print(f"Prediction    : {r2.json()}")

# ── Batch test: 5 different inputs ────────────────────────────────────────────
test_inputs = [
    {"MedInc": 8.5,  "HouseAge": 10, "AveRooms": 7.2, "AveBedrms": 1.0,
     "Population": 900,  "AveOccup": 2.8, "Latitude": 37.8, "Longitude": -122.4},
    {"MedInc": 2.1,  "HouseAge": 40, "AveRooms": 4.1, "AveBedrms": 1.3,
     "Population": 2100, "AveOccup": 3.8, "Latitude": 34.0, "Longitude": -118.2},
    {"MedInc": 11.0, "HouseAge": 5,  "AveRooms": 8.9, "AveBedrms": 1.0,
     "Population": 600,  "AveOccup": 2.5, "Latitude": 37.4, "Longitude": -122.1},
    {"MedInc": 1.5,  "HouseAge": 52, "AveRooms": 3.8, "AveBedrms": 1.5,
     "Population": 3200, "AveOccup": 4.2, "Latitude": 34.1, "Longitude": -117.9},
    {"MedInc": 6.3,  "HouseAge": 18, "AveRooms": 5.5, "AveBedrms": 1.1,
     "Population": 1400, "AveOccup": 3.0, "Latitude": 36.7, "Longitude": -121.6},
]

print()
print(f"  {'MedInc':>8}  {'HouseAge':>9}  {'Prediction':>20}")
print("  " + "-" * 42)
for inp in test_inputs:
    r = httpx.post(f"{RENDER_URL}/predict", json=inp)
    pred = r.json().get("predicted_house_value_display", "error")
    print(f"  {inp['MedInc']:>8.1f}  {inp['HouseAge']:>9.0f}  {pred:>20}")

### 📝 Exercise B.4 — Deploy and test from a notebook

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise B.4 -- Deploy and explore your live endpoint
#
# After your Render deployment is live:
#
# 1. Replace RENDER_URL above with your actual URL and run the cells.
#    Confirm you get sensible predictions back.
#
# 2. Test the automatic validation:
#    Send a request with a missing field and confirm you get a 422 back.
#    Send a request where Population is a string ("abc") -- what status code returns?
#
# 3. Measure latency:
#    import time
#    start = time.time()
#    httpx.post(f"{RENDER_URL}/predict", json=sample)
#    print(f"Response time: {time.time()-start:.2f}s")
#    Note: the free Render tier "sleeps" after 15 minutes idle.
#    The first request after sleep takes ~30 seconds (a "cold start").
#    Subsequent requests are fast. This is normal for free-tier deployments.
#
# 4. Open your Render URL in a mobile browser.
#    You will see the raw JSON -- the model is accessible from any device.
# ─────────────────────────────────────────────────────────────────────────────

print("Exercise B.4 -- deploy to Render, then run tests from this notebook.")

---
# B.5 Updating a Deployed Model

## When to update

Models go stale. New data arrives. Performance drifts. Your team retrains.

When this happens, you need to:
1. Produce a new `.pth` file
2. Update the deployed service to use it
3. Confirm the predictions actually changed

The key constraint: **do this without breaking anything that is already calling your endpoint.**

---

## The update workflow

```
Notebook                    GitHub             Render
────────                    ──────             ──────
retrain pipeline
     |
save as v2.pth
     |
update app.py  ──── push ──> new commit ──> Render detects push
     |                                           |
copy v2.pth    ──── push ──>                 rebuilds container
                                                 |
                                             restarts service
                                                 |
test endpoint  <───────────────────────────  live with v2
```

**Render rebuilds and restarts automatically on every GitHub push.**
No manual server management required.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1 -- Retrain and save the updated pipeline
#
# In practice, new_loader comes from fresh data collected since the last training.
# Here we simulate it using the validation set -- the pattern is identical.
# ─────────────────────────────────────────────────────────────────────────────

import torch.optim as optim

# Load the currently deployed pipeline
pipeline_v1 = ModelPipeline.load("california_pipeline_v1.pth")
print("v1 loaded.")
print(f"  Epochs in history : {len(pipeline_v1.training_history.get('val_loss', []))}")

# Retrain for 15 additional epochs on new data
retrain_opt = optim.Adam(pipeline_v1.model.parameters(), lr=5e-4, weight_decay=1e-4)

print("\nRetraining on new data (15 epochs)...")
pipeline_v1.retrain(
    train_loader = new_loader,    # your updated DataLoader
    val_loader   = val_loader,
    criterion    = criterion,
    optimiser    = retrain_opt,
    num_epochs   = 15,
    device       = device,
)

# Save as v2 -- NEVER overwrite the previous version
pipeline_v1.save("california_pipeline_v2.pth")
print(f"\nSaved: california_pipeline_v2.pth")
print(f"  Total epochs in history : {len(pipeline_v1.training_history.get('val_loss', []))}") 

## Step 2 — Update `app.py` and push to GitHub

Change exactly **one line** in `app.py`:

```python
# Before
pipeline = ModelPipeline.load("california_pipeline_v1.pth")

# After
pipeline = ModelPipeline.load("california_pipeline_v2.pth")
```

Then push the new `.pth` file and the updated `app.py` to GitHub:

```powershell
git add california_pipeline_v2.pth app.py
git commit -m "Deploy v2: retrained on new data"
git push
```

Render detects the push within seconds, rebuilds the container, and restarts the service.
The process takes 3–5 minutes. When the Render dashboard shows "Live" again, the new
model is serving requests.

---

## Step 3 — Confirm predictions changed

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 -- Compare v1 and v2 predictions
#
# After the redeployment is complete, run this cell to confirm that
# the live endpoint is now returning different predictions.
#
# We compare v1 (loaded locally) against v2 (live on Render).
# ─────────────────────────────────────────────────────────────────────────────

import httpx, numpy as np

RENDER_URL = "https://house-predictor-xxxx.onrender.com"   # <- your Render URL

test_cases = [
    {"MedInc": 5.2, "HouseAge": 25, "AveRooms": 6.1, "AveBedrms": 1.1,
     "Population": 1200, "AveOccup": 3.1, "Latitude": 34.2, "Longitude": -118.4},
    {"MedInc": 2.1, "HouseAge": 40, "AveRooms": 4.1, "AveBedrms": 1.3,
     "Population": 2100, "AveOccup": 3.8, "Latitude": 34.0, "Longitude": -118.2},
]

print(f"  {'Input':>6}  {'v1 (local)':>18}  {'v2 (live)':>18}  {'Changed?':>10}")
print("  " + "-" * 58)

for i, inp in enumerate(test_cases):
    # v1 prediction -- run locally
    X = np.array([[inp["MedInc"], inp["HouseAge"], inp["AveRooms"], inp["AveBedrms"],
                   inp["Population"], inp["AveOccup"], inp["Latitude"], inp["Longitude"]]])
    raw_v1  = pipeline_v1.predict(X)
    pred_v1 = f"${float(np.expm1(raw_v1[0][0]) * 100_000):,.0f}"

    # v2 prediction -- call the live endpoint
    r = httpx.post(f"{RENDER_URL}/predict", json=inp)
    pred_v2 = r.json().get("predicted_house_value_display", "error")

    changed = "Yes" if pred_v1 != pred_v2 else "No -- check redeploy"
    print(f"  Case {i+1}  {pred_v1:>18}  {pred_v2:>18}  {changed:>10}")

## Version naming convention

| File | When to create |
|------|---------------|
| `california_pipeline_v1.pth` | Initial training (Chapter 3) |
| `california_pipeline_v2.pth` | First retrain on new data |
| `california_pipeline_v3.pth` | Architecture change or major retraining |

**Never delete or overwrite an existing `.pth` file.**
If `v2` produces worse predictions than `v1`, rolling back is one line in `app.py`
and one `git push`. Render redeploys automatically.

> **This is the same versioning discipline you have practised since Chapter 3.**
> Every `.pth` file is a checkpoint. Every deployment is reproducible.
> The pattern is identical whether you are in a notebook, a Docker container,
> or a cloud server.

### 📝 Exercise B.5 — Retrain, redeploy, and verify

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise B.5 -- Full update cycle
#
# Walk through the complete update workflow from start to finish.
#
# 1. Retrain the Chapter 3 pipeline for 10 more epochs (any learning rate).
#    Save the result as california_pipeline_v2.pth.
#
# 2. Update the one line in app.py that loads the model.
#
# 3. Push both files to GitHub.
#    Watch the Render dashboard -- confirm it detects the push and rebuilds.
#
# 4. Run the comparison cell above.
#    Do the v1 and v2 predictions differ?
#    If they are identical, the retrain did not change the weights much --
#    that is not necessarily a problem, but good to know.
#
# 5. Roll back: change app.py back to v1, push, and confirm the live
#    predictions return to their original values.
#
# Reflection:
#   - In a production system, how would you decide whether v2 is good enough
#     to replace v1? What metric would you check before pushing?
# ─────────────────────────────────────────────────────────────────────────────

print("Exercise B.5 -- retrain, redeploy, verify, and optionally roll back.")

---
## Bonus Chapter Summary

| Concept | Key takeaway |
|---------|-------------|
| **The gap** | A notebook requires a human; a deployed model serves other systems automatically |
| **FastAPI** | Turns a Python function into a web endpoint; Pydantic validates inputs before your code runs |
| **`/predict` endpoint** | Accepts JSON → runs the ModelPipeline → returns JSON; handles one input at a time |
| **`/` health check** | A simple GET that returns `{"status":"ok"}`; cloud platforms use this to monitor your service |
| **422 Unprocessable Entity** | FastAPI's automatic rejection of malformed requests — bad data never reaches the model |
| **Docker** | Packages your code, model, Python version, and all libraries into a portable sealed container |
| **Dockerfile** | A four-step recipe: base image → install packages → copy files → run server |
| **Render** | Free cloud platform; connect GitHub → auto-deploy on every push; no server management |
| **Updating** | Save new `.pth` → change one line in `app.py` → push → Render rebuilds automatically |
| **Versioning** | Never overwrite `.pth` files; rollback is one line change and one `git push` |

---

## The Complete Journey

```
Chapter 1  -->  A single neuron learns to fit a line
Chapter 2  -->  NumPy, pandas, PyTorch tensors -- the building blocks
Chapter 3  -->  Training, saving, deploying: the ModelPipeline
Chapter 4  -->  Classifying tabular data: customer churn
Chapter 5  -->  Reading images: CNN for defect detection
Chapter 6  -->  Forecasting sequences: LSTM for CO2 demand
Chapter 7  -->  Directing pre-trained LLMs: earnings analyst bot
Bonus      -->  Taking any model live: FastAPI + Docker + Cloud
```

Every model built in this book can follow the path in this chapter —
from a `.pth` file in a notebook to a live API endpoint that any team
or application can build products on top of.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*
*Dr. M. Ramasubramaniam & Mr. Daniel Peter*